# NBA Draft vs Rookie Season Stats
Pulls NBA per-game stats and draft history from Kaggle, then merges them to isolate each player's rookie season.

## Imports

In [1]:
%pip install kagglehub 
%pip install pandas

import kagglehub
from kagglehub import KaggleDatasetAdapter


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached pandas-3.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
Using cached pandas-3.0.3-cp312-cp312-macosx_11_0_arm64.whl (9.9 MB)

[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/kaelanbrose/Desktop/All Folders/VS Code/Projects/AP Stat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [ ]:
import pandas as pd

PerGame_Path = kagglehub.dataset_download('sumitrodatta/nba-aba-baa-stats', path='Player Per Game.csv')
Draft_Path = kagglehub.dataset_download('sumitrodatta/nba-aba-baa-stats', path='Draft Pick History.csv')

PerGame_df = pd.read_csv(PerGame_Path)
Draft_df = pd.read_csv(Draft_Path)

print("Number of rows (Per Game):", len(PerGame_df))
print("Number of rows (Draft):", len(Draft_df))

Number of rows (Per Game): 33339
Number of rows (Draft): 8383


## Merge & Filter to Rookie Seasons

In [20]:
# Rename draft season to avoid column collision during merge
Draft_df = Draft_df.rename(columns={'season': 'draft_season'})

# Merge on player_id, then filter to only the rookie season (stats season == draft season + 1)
merged_df = pd.merge(PerGame_df, Draft_df, on='player_id')
rookie_df = merged_df[merged_df['season'] == merged_df['draft_season'] + 1].reset_index(drop=True)
print("Columns: ", rookie_df.columns)

#Filters columns we want 
rookie_df_filtered = rookie_df[["draft_season", "overall_pick", "player_x", "team", "pos", "pts_per_game"]]

#Sorts by pts per game and prints
print(rookie_df_filtered.sort_values(by="pts_per_game", ascending=False))


Columns:  Index(['season', 'lg_x', 'player_x', 'player_id', 'age', 'team', 'pos', 'g',
       'gs', 'mp_per_game', 'fg_per_game', 'fga_per_game', 'fg_percent',
       'x3p_per_game', 'x3pa_per_game', 'x3p_percent', 'x2p_per_game',
       'x2pa_per_game', 'x2p_percent', 'e_fg_percent', 'ft_per_game',
       'fta_per_game', 'ft_percent', 'orb_per_game', 'drb_per_game',
       'trb_per_game', 'ast_per_game', 'stl_per_game', 'blk_per_game',
       'tov_per_game', 'pf_per_game', 'pts_per_game', 'draft_season', 'lg_y',
       'overall_pick', 'round', 'tm', 'player_y', 'college'],
      dtype='str')
      draft_season  overall_pick          player_x team  pos  pts_per_game
3300          1959           3.0  Wilt Chamberlain  PHW    C          37.6
2707          1972          12.0     Julius Erving  VIR   SF          31.9
3252          1961           1.0      Walt Bellamy  CHP    C          31.6
3293          1960           1.0   Oscar Robertson  CIN   PG          30.5
2861          1970       

## Split into Lottery and Late First Round Pick Tables

In [29]:
lottery_df = rookie_df_filtered[rookie_df_filtered["overall_pick"] <= 14]
print(lottery_df)

lateRd_df = rookie_df_filtered[(rookie_df_filtered["overall_pick"] >= 15) & (rookie_df_filtered["overall_pick"] <= 30)]
print(lateRd_df)

      draft_season  overall_pick       player_x team  pos  pts_per_game
0             2025           5.0     Ace Bailey  UTA   SF          13.8
5             2025          14.0  Carter Bryant  SAS   PF           4.2
10            2025          11.0  Cedric Coward  MEM   SG          13.6
11            2025           8.0     Egor Dёmin  BRK   PG          10.3
13            2025           3.0   VJ Edgecombe  PHI   SG          16.0
...            ...           ...            ...  ...  ...           ...
3639          1947           6.0  Chink Crossin  PHW  NaN           1.8
3640          1947           3.0   Bulbs Ehlers  BOS  NaN           7.2
3645          1947           5.0     Dick Holub  NYK  NaN          10.5
3647          1947           8.0    Paul Huston  CHS  NaN           3.6
3650          1947           9.0   Dick O'Keefe  WSC  NaN           4.2

[1074 rows x 6 columns]
      draft_season  overall_pick        player_x team  pos  pts_per_game
2             2025          17.0   Joa

## Sample from the datasets 

In [37]:
# 10% sample from each table
lottery_sample = lottery_df.sample(frac=0.1)
lateRd_sample = lateRd_df.sample(frac=0.1)

print(lottery_sample)
print(lateRd_sample)

      draft_season  overall_pick        player_x team pos  pts_per_game
2312          1979          14.0    Brad Holland  LAL  SG           2.8
2415          1977          11.0  Ernie Grunfeld  MIL  SG           6.9
1522          1995          11.0      Gary Trent  POR  PF           7.5
1992          1985          10.0     Ed Pinckney  PHO  SF           8.5
2657          1973          12.0   Kevin Kunnert  BUF   C           2.8
...            ...           ...             ...  ...  ..           ...
1260          2000          10.0   Keyon Dooling  LAC  PG           5.9
1737          1990          13.0      Loy Vaught  LAC  PF           5.5
1728          1990           2.0     Gary Payton  SEA  PG           7.2
2566          1974           3.0    Tom Burleson  SEA   C          10.1
3151          1966           5.0      Jack Marin  BAL  SF           9.6

[107 rows x 6 columns]
      draft_season  overall_pick       player_x team  pos  pts_per_game
1792          1988          30.0   Fenni

## Print the data

In [38]:
print("Mean for lottery: ", lottery_sample["pts_per_game"].mean())
print("Standard Deviation for lottery: ", lottery_sample["pts_per_game"].std())

print("Mean for late round: ", lateRd_sample["pts_per_game"].mean())
print("Standard Deviation for late round: ", lateRd_sample["pts_per_game"].std())

Mean for lottery:  9.176635514018692
Standard Deviation for lottery:  5.0682433308037105
Mean for late round:  5.452941176470588
Standard Deviation for late round:  4.248836902754506
